In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import pytz

In [ ]:
local_tz = pytz.timezone('Asia/Jakarta')  # Replace with your time zone
now_local = datetime.now(local_tz)
date_format = "%m-%d-%Y" # Choose your desired format
date_str = now_local.strftime(date_format)

In [ ]:
from google.colab import files
file = files.upload()

In [ ]:
# Get the filename from the uploaded file dictionary
uploaded_filename = list(file.keys())[0]

df = pd.read_csv(uploaded_filename)
df_backup = df.copy(deep=True)

In [ ]:
df_columns = ['delivery_date',  'hubs', 'time_slot', 'total_weight_perorder', 'order_no']

df_filtered = df[df_columns].copy() # Create a copy to avoid SettingWithCopyWarning

# Convert 'delivery_date' to datetime objects
df_filtered['delivery_date'] = pd.to_datetime(df_filtered['delivery_date'], errors='coerce')


# Add 'week' and 'month' columns
df_filtered.loc[:, 'week'] = df_filtered['delivery_date'].dt.isocalendar().week
df_filtered.loc[:, 'month'] = df_filtered['delivery_date'].dt.month

df_filtered.loc[:, 'time_slot'] = df_filtered['time_slot'].str.replace('slot-12bb', 'slot-1')
df_filtered.loc[:, 'time_slot'] = df_filtered['time_slot'].str.replace('slot-1bb', 'slot-1')
df_filtered.loc[:, 'time_slot'] = df_filtered['time_slot'].str.replace('slot-0bb', 'slot-0')
df_filtered.loc[:, 'time_slot'] = df_filtered['time_slot'].str.replace('slot-b2b-2', 'slot-0')
df_filtered.loc[:, 'time_slot'] = df_filtered['time_slot'].str.replace('slot-13', 'slot-sameday')
df_filtered.loc[:, 'time_slot'] = df_filtered['time_slot'].str.replace('slot-sameday03', 'slot-2')
# Sort by multiple columns:
sorted_df = df_filtered.sort_values(by=['hubs', 'delivery_date', 'time_slot'], ascending=[True, True,True])

sorted_df.head()

In [ ]:
# Create the pivot table
pivot_table = sorted_df.pivot_table(values=['total_weight_perorder', 'order_no'], index=['delivery_date', 'hubs', 'week', 'month'], columns='time_slot', aggfunc={'total_weight_perorder': 'sum', 'order_no': 'count'})

# Reset the index to make 'week' and 'month' regular columns
pivot_table_reset = pivot_table.reset_index()

# Flatten the MultiIndex columns more carefully
new_column_names = []
for col in pivot_table_reset.columns:
    if isinstance(col, tuple):
        # Join non-empty parts of the tuple with an underscore
        # This handles cases like ('delivery_date', '') becoming 'delivery_date'
        cleaned_name = '_'.join(filter(None, col))
        new_column_names.append(cleaned_name)
    else:
        new_column_names.append(col) # Keep single-level columns as is

pivot_table_reset.columns = new_column_names

# Get the list of columns
cols = pivot_table_reset.columns.tolist()

# Identify the columns to move (these should now be 'week' and 'month' without underscores)
cols_to_move = ['week', 'month']
cols_to_keep = [col for col in cols if col not in cols_to_move]

# Create the new ordered list of columns
new_order = cols_to_keep + cols_to_move

# Reindex the DataFrame with the new column order
pivot_table_ordered = pivot_table_reset[new_order]

# Set the index back to delivery_date and hubs (using original names)
pivot_table = pivot_table_ordered.set_index(['delivery_date', 'hubs'])

# Print the pivot table
pivot_table

In [ ]:
pivot_table.shape

In [ ]:
pivot_table.to_csv('fleet_capacity_utilization.csv', index=True, header=False)
files.download('fleet_capacity_utilization.csv')